## Task 1.1 **Password Exercises**

In this exercise we will use Python to create and crack password hashes. The exercise is written with the assumption that we are in the University's Linux Computer lab. If you are using your own computer, you may have to install additional software.

Use the Python hashlib library to hash the password "HelloWorld" with the SHA1 hash function.

The Python program does not need to be fancy at this point, we can hardcode the string "HelloWorld" into the program.

The following page describes hashlib and gives some usage examples.
https://docs.python.org/3/library/hashlib.html

Print out the hash value as a hex string and compare it with the output produced by the sha1sum binary. Run
**echo -n "HelloWorld" | sha1sum** in the terminal.
Since hash functions are deterministic, results should match! Observe the -n command line flag which ensures that echo does not produce a newline character after the HelloWorld string.

In [1]:
import hashlib

password = "HelloWorld"
# Use encode() to get bytes (equivalent to echo -n: no newline)
digest = hashlib.sha1(password.encode()).hexdigest()
print("SHA1 hash (hex):", digest)

SHA1 hash (hex): db8ac1c259eb89d4a131b253bacfca5f319d54f2


**Verify:** Run in terminal: `echo -n "HelloWorld" | sha1sum`  
Expected hash: `db8ac1c259eb89d4a131b253bacfca5f319d54f2` — should match the Python output above.

## Task 1.2 SHA1

Read hashcat's example_hashes page: https://hashcat.net/wiki/doku.php?id=example_hashes

Use the python code from above to produce the hash values shown for the SHA1 password hash variants 100, 110, and 120.

From [hashcat example_hashes](https://hashcat.net/wiki/doku.php?id=example_hashes):
- **100** = SHA1(pass) — plain SHA1 (same as task 1)

- **110** = sha1(pass, salt) — password then salt; example salt `2014`

- **120** = sha1(salt, pass) — salt then password; example salt `5363620024`

Use the **same password "HelloWorld"** and the same hashlib approach to produce these three variant hashes.

In [2]:
import hashlib

# Same password as task 1.1
password = "HelloWorld"

# 100: SHA1($pass) — plain SHA1
h100 = hashlib.sha1(password.encode()).hexdigest()
print(f"100  SHA1($pass):              {h100}")

# 110: sha1($pass.$salt) — password + salt; use example salt from hashcat
salt_110 = "2014"
h110 = hashlib.sha1((password + salt_110).encode()).hexdigest()
print(f"110  sha1($pass.$salt):        {h110}:{salt_110}")

# 120: sha1($salt.$pass) — salt + password; use example salt from hashcat
salt_120 = "5363620024"
h120 = hashlib.sha1((salt_120 + password).encode()).hexdigest()
print(f"120  sha1($salt.$pass):        {h120}:{salt_120}")

100  SHA1($pass):              db8ac1c259eb89d4a131b253bacfca5f319d54f2
110  sha1($pass.$salt):        0efbe5678c6d20a6abebd85a6b12d393dc657143:2014
120  sha1($salt.$pass):        85a04ce58b84c1c359734b6c3951effd1b7d1289:5363620024


## Task 1.3 Break SHA1

Break the SHA1 hash (no salt) of any 6 digit PIN when given the PIN's hash value.


**Find the PIN for:** `b498bfa2498e21325d1178417bea459eb2cd28f8`

In [3]:
import hashlib
from typing import Optional

def crack_pin(target_hash: str) -> Optional[str]:
    """Try all 6-digit PINs (000000..999999), return PIN if SHA1 matches target_hash (no salt)."""
    target = target_hash.strip().lower().split()[0]  # ignore trailing newline/sha1sum filename
    for i in range(0, 1000000):
        pin = "{:06d}".format(i)
        if hashlib.sha1(pin.encode()).hexdigest() == target:
            return pin
    return None

# (In notebook we run the next cell to test and find the target PIN.)

In [4]:
# Test: hash of "000123" should be 9bcea4483a009e15aa649981c183c5819dd9e185
test_hash = "9bcea4483a009e15aa649981c183c5819dd9e185"
found = crack_pin(test_hash)
print(f"Test (hash of 000123): crack_pin({test_hash[:20]}...) -> {found}")
assert found == "000123", "Test failed"

# Find PIN for the given hash
target_hash = "b498bfa2498e21325d1178417bea459eb2cd28f8"
pin = crack_pin(target_hash)
print(f"Target hash {target_hash} -> PIN: {pin}")

Test (hash of 000123): crack_pin(9bcea4483a009e15aa64...) -> 000123
Target hash b498bfa2498e21325d1178417bea459eb2cd28f8 -> PIN: 314159


## Task 1.4. HMAC-SHA1

Refer to the Python hmac library documentation at https://docs.python.org/3/library/hmac.html and produce the hash values for the hashcat example hashes 150 and 160.

Mode 150: HMAC-SHA1 with key=hashcat, msg=1234. 

Mode 160: HMAC-SHA1 with key=1234, msg=hashcat. 

Use **hmac.digest(key, msg, digest)** and output hex to match hashcat format.

In [5]:
import hmac

# Hashcat 150: HMAC-SHA1(key=$pass, msg=salt). password=hashcat, salt=1234
key_150 = b"hashcat"
msg_150 = b"1234"
digest_150 = hmac.digest(key_150, msg_150, "sha1")
hash_150 = digest_150.hex()
print("Mode 150:", f"{hash_150}:1234")
assert hash_150 == "c898896f3f70f61bc3fb19bef222aa860e5ea717", "150 mismatch"

# Hashcat 160: HMAC-SHA1(key=$salt, msg=$pass). salt=1234, password=hashcat
key_160 = b"1234"
msg_160 = b"hashcat"
digest_160 = hmac.digest(key_160, msg_160, "sha1")
hash_160 = digest_160.hex()
print("Mode 160:", f"{hash_160}:1234")
assert hash_160 == "d89c92b4400b15c39e462a8caa939ab40c3aeeea", "160 mismatch"

Mode 150: c898896f3f70f61bc3fb19bef222aa860e5ea717:1234
Mode 160: d89c92b4400b15c39e462a8caa939ab40c3aeeea:1234


## Task 2.1. TOTP exercise

In this exercise, we aim to create a QR code using a python library called "qrcode".

Run the following cells.

In [6]:
!pip3 install qrcode
!pip install image

In [7]:
import qrcode

img = qrcode.make('Hello World')
img.save('qrcode.png')
print("Saved qrcode.png")

Saved qrcode.png


## Task 2.2 Generate valid TOTP QR codes (OATH URI format)

Read https://docs.yubico.com/yesdk/users-manual/application-oath/uri-string-format.html first.

Now, we will produce valid TOTP QR codes with our own secret key.

See [Yubico OATH URI format](https://docs.yubico.com/yesdk/users-manual/application-oath/uri-string-format.html):

- **Scheme**: `otpauth://`
- **Type**: `totp`
- **Label**: `Issuer:Account` (e.g. `COM2041:user@example.com`), URL-encode if needed
- **Parameters**: `secret` (Base32, no padding), optional `issuer`, `algorithm`, `digits`, `period`

Format: `otpauth://totp/LABEL?secret=BASE32_SECRET&issuer=Issuer&period=30&digits=6`

In [8]:
import qrcode
from urllib.parse import quote

def make_totp_uri(secret_b32: str, account: str, issuer: str = "COM2041") -> str:
    """Build otpauth URI for TOTP. Secret is Base32 (RFC 3548), padding optional."""
    label = f"{issuer}:{account}" if issuer else account
    params = [
        f"secret={secret_b32}",
        f"issuer={quote(issuer)}" if issuer else None,
        "algorithm=SHA1",
        "digits=6",
        "period=30",
    ]
    query = "&".join(p for p in params if p)
    return f"otpauth://totp/{quote(label)}?{query}"

# Your chosen secret (Base32, no padding)
MY_SECRET_B32 = "JBSWY3DPEHPK3PXP"  # replace with your own if want to
account_name = "student@example.com"
uri = make_totp_uri(MY_SECRET_B32, account_name)
print("TOTP URI:", uri)

img = qrcode.make(uri)
img.save("totp_qrcode.png")
print("Saved totp_qrcode.png — scan with your authenticator app to add this account.")

TOTP URI: otpauth://totp/COM2041%3Astudent%40example.com?secret=JBSWY3DPEHPK3PXP&issuer=COM2041&algorithm=SHA1&digits=6&period=30
Saved totp_qrcode.png — scan with your authenticator app to add this account.


## Task 2.3. Generate 6-digit TOTP as your Authenticator app

We write a Python program in the following cell to produce the same codes as your Authenticator app for the secret key that you chose. You will need to read the following RFC for the overall TOTP code generation https://www.rfc-editor.org/rfc/rfc6238 and the following RFC for the specification and test values of the truncate function: https://www.rfc-editor.org/rfc/rfc4226. Note that you do not need to understand RFC4226/6238 in detail.

We will also need to import the Python datetime library to count the number of 30-second intervals since 1/1/1970: from datetime import datetime, UTC and probably the base64 library to access the base32 decode function. Note that you can simply treat base32 decode function as black-box.

**RFC 4226**: HOTP(K,C) = Truncate(HMAC-SHA1(K,C)). **Truncate**: from the 20-byte HMAC result, take the low 4 bits of the last byte as offset; extract 4 bytes at that offset, mask the first byte with 0x7f; convert to integer; OTP = that value mod 10^6 (6 digits). Key point is that "4-byte (32-bit string) converts to 6-digits using mod 10^6".

**RFC 6238**: TOTP = HOTP(K, T) where **T** = floor((current Unix time − T0) / X), T0=0, X=30 seconds.

Key K is the Base32-decoded secret (add padding when decoding if needed).

In [9]:
import hmac
import base64
from datetime import datetime, timezone

def base32_decode_no_pad(secret_b32: str) -> bytes:
    """Decode Base32 (RFC 3548); add padding if needed."""
    s = secret_b32.upper().replace(" ", "")
    pad = (8 - len(s) % 8) % 8
    return base64.b32decode(s + "=" * pad)

def hotp_truncate(hmac_result: bytes, digits: int = 6) -> int:
    """RFC 4226: extract 4 bytes then mod 10^6."""
    if len(hmac_result) != 20:
        raise ValueError("HMAC-SHA1 must be 20 bytes")
    offset = hmac_result[19] & 0xF
    """  """
    bin_code = (
        (hmac_result[offset] & 0x7F) << 24
        | (hmac_result[offset + 1] & 0xFF) << 16
        | (hmac_result[offset + 2] & 0xFF) << 8
        | (hmac_result[offset + 3] & 0xFF)
    )
    return bin_code % (10**digits)

def totp(secret_b32: str, t0: int = 0, period: int = 30, digits: int = 6) -> str:
    """Compute 6-digit TOTP for current time. secret_b32 is Base32 key."""
    key = base32_decode_no_pad(secret_b32)
    now = datetime.now(timezone.utc)
    unix_time = int(now.timestamp())
    T = (unix_time - t0) // period
    counter = T.to_bytes(8, "big")  # RFC 4226: 8-byte big-endian counter
    h = hmac.digest(key, counter, "sha1") # To do by the students!!!
    code = hotp_truncate(h, digits) # To do by the students
    return str(code).zfill(digits)

# Same secret as the one in the QR code
MY_SECRET_B32 = "JBSWY3DPEHPK3PXP"
code = totp(MY_SECRET_B32) # To do by the students
print("Current TOTP (6 digits):", code) # To do by the students
print("Compare with the authenticator entry added from totp_qrcode.png.")

Current TOTP (6 digits): 301070
Compare with the authenticator entry added from totp_qrcode.png.


## Task 2.4

Finally, we verify RFC 4226 truncation example and RFC 6238 test vector. This gives us confidence our code is doing what it should be doing. You can run this cell alone. The parameters refer to RFC 4226 https://www.rfc-editor.org/rfc/rfc4226. and RFC 6238 https://www.rfc-editor.org/rfc/rfc6238.

In [10]:
import hmac

def hotp_truncate(hmac_result: bytes, digits: int = 6) -> int:
    """RFC 4226: extract 4 bytes then mod 10^6."""
    if len(hmac_result) != 20:
        raise ValueError("HMAC-SHA1 must be 20 bytes")
    offset = hmac_result[19] & 0xF
    bin_code = (
        (hmac_result[offset] & 0x7F) << 24
        | (hmac_result[offset + 1] & 0xFF) << 16
        | (hmac_result[offset + 2] & 0xFF) << 8
        | (hmac_result[offset + 3] & 0xFF)
    )
    return bin_code % (10**digits)

# RFC 4226 example: last byte 0x5a -> offset=10 -> HOTP = 872921. These parameters refer to "https://datatracker.ietf.org/doc/html/rfc4226"
rfc4226_hmac = bytes([
    0x1f, 0x86, 0x98, 0x69, 0x0e, 0x02, 0xca, 0x16, 0x61, 0x85, 0x50, 0xef,
    0x7f, 0x19, 0xda, 0x8e, 0x94, 0x5b, 0x55, 0x5a,
])
assert hotp_truncate(rfc4226_hmac, 6) == 872921
print("RFC 4226 truncation example: 872921 ✓")

# RFC 6238: secret "12345678901234567890" (ASCII), time 59, T=1, 8-digit TOTP = 94287082. These parameters refer to "https://datatracker.ietf.org/doc/html/rfc6238"
secret_ascii = b"12345678901234567890"
T = 1
counter = T.to_bytes(8, "big")
h = hmac.digest(secret_ascii, counter, "sha1")
totp8 = hotp_truncate(h, 8)
assert totp8 == 94287082, f"Expected 94287082, got {totp8}"
print("RFC 6238 test (time=59, 8 digits): 94287082 ✓")

RFC 4226 truncation example: 872921 ✓
RFC 6238 test (time=59, 8 digits): 94287082 ✓
